In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("\n--- NGÀY 92: MÁY BĂM TÀI LIỆU PDF (DOCUMENT LOADERS & SPLITTERS) ---")

#1. Tải tài liệu PDF
pdf_path = "tai_lieu.pdf"

try:
    print(f"[1] Đang tải tài liệu PDF từ: {pdf_path}")
    loader = PyPDFLoader(pdf_path)

    #Load() sẽ biến mỗi trang sách thành 1 Object gọi là 'Document'
    documents = loader.load()
    print(f"=> Tài liệu có {len(documents)} trang.")

    #2.Băm nhỏ tài liệu (CHUNKING)
    print("\n[2] Đang băm nhỏ tài liệu thành các đoạn văn bản...")
    
    may_bam = RecursiveCharacterTextSplitter(
        chunk_size =1500,
        chunk_overlap=300,
        length_function = len
    )

    #Bắt đầu cắt
    cac_mau_doan_van = may_bam.split_documents(documents)
    print(f"=> Băm thành công! Từ {len(documents)} trang sách ban đầu, ta đã cắt được {len(cac_mau_doan_van)} mẩu nhỏ (Chunks).")

    print(f"Xem một mẩu nhỏ (Chunk) mẫu:")
    print(f"{cac_mau_doan_van[0].page_content}")
    print("---------------------------------------------")
    print(f"\n[Metadata]: Mẩu này được trích từ file '{cac_mau_doan_van[0].metadata['source']}', trang {cac_mau_doan_van[0].metadata['page']+1}")

except FileNotFoundError:
    print(f"\n[LỖI]: Không tìm thấy file '{pdf_path}'. Bạn nhớ copy file vào thư mục và đổi tên cho đúng nhé!")
except Exception as e:
    print(f"\n[LỖI HỆ THỐNG]: {e}")

c:\Users\ACER\anaconda3\envs\jarvis_ai\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(
c:\Users\ACER\anaconda3\envs\jarvis_ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- NGÀY 92: MÁY BĂM TÀI LIỆU PDF (DOCUMENT LOADERS & SPLITTERS) ---
[1] Đang tải tài liệu PDF từ: tai_lieu.pdf
=> Tài liệu có 34 trang.

[2] Đang băm nhỏ tài liệu thành các đoạn văn bản...
=> Băm thành công! Từ 34 trang sách ban đầu, ta đã cắt được 44 mẩu nhỏ (Chunks).
Xem một mẩu nhỏ (Chunk) mẫu:
A practical  
guide to  
building agents
---------------------------------------------

[Metadata]: Mẩu này được trích từ file 'tai_lieu.pdf', trang 1


In [2]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import time

print("\n--- NGÀY 93: MÃ HÓA VECTOR VÀ LƯU VÀO CHROMADB (PHIÊN BẢN MIỄN PHÍ) ---")

# 1. Khởi tạo mô hình Embedding của HuggingFace (Chạy offline trên máy anh)
print("[1] Đang khởi tạo mô hình Embedding HuggingFace...")
# Model này rất nhẹ (khoảng 80MB) và cực kỳ ổn định cho tiếng Việt
may_ma_hoa = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Tạo cơ sở dữ liệu
# Em đề xuất anh đổi tên thư mục database thành 'database_groq' để tránh xung đột với đồ cũ
database_path = r"f:\Neuro-sama\Jarvis_Project\notebooks\Phase2\database_groq"

print(f"[2] Bắt đầu nạp {len(cac_mau_doan_van)} mẩu vào ChromaDB...")

# Xóa database cũ nếu anh muốn làm mới hoàn toàn (tùy chọn)
# if os.path.exists(database_path):
#     import shutil
#     shutil.rmtree(database_path)

vector_db = Chroma.from_documents(
    documents=cac_mau_doan_van,
    embedding=may_ma_hoa,
    persist_directory=database_path 
)

print(f"=> Hoàn tất! Đã lưu Database vào: {database_path}")

# 3. Kiểm tra lại bằng cách truy vấn (Similarity Search)
cau_hoi_test = "Agent là gì và làm sao để xây dựng nó?"
print(f"\n[3] Bắt đầu tìm kiếm thử nghiệm: '{cau_hoi_test}'")

# Truy vấn thử xem database có hoạt động không
ket_qua = vector_db.similarity_search(cau_hoi_test, k=3)

if len(ket_qua) > 0:
    print(f"\n=> THÀNH CÔNG! Tìm thấy {len(ket_qua)} đoạn văn bản liên quan nhất:")
    for i, doc in enumerate(ket_qua):
        print(f"\n--- ĐOẠN {i+1} (Trang {doc.metadata.get('page', 0) + 1}) ---")
        # In ra 200 ký tự đầu để kiểm tra nội dung
        print(doc.page_content[:200] + "...")
else:
    print("\n=> [CẢNH BÁO]: Không tìm thấy kết quả nào. Anh hãy kiểm tra lại biến 'cac_mau_doan_van' nhé!")


--- NGÀY 93: MÃ HÓA VECTOR VÀ LƯU VÀO CHROMADB (PHIÊN BẢN MIỄN PHÍ) ---
[1] Đang khởi tạo mô hình Embedding HuggingFace...


c:\Users\ACER\anaconda3\envs\jarvis_ai\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ACER\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5123.80it/s]
BertModel LOAD REPORT f

[2] Bắt đầu nạp 44 mẩu vào ChromaDB...
=> Hoàn tất! Đã lưu Database vào: f:\Neuro-sama\Jarvis_Project\notebooks\Phase2\database_groq

[3] Bắt đầu tìm kiếm thử nghiệm: 'Agent là gì và làm sao để xây dựng nó?'

=> THÀNH CÔNG! Tìm thấy 3 đoạn văn bản liên quan nhất:

--- ĐOẠN 1 (Trang 18) ---
M anager  pa tt ern
The manager  pa tt ern empo w er s a cen tr al LLM— the “ manager” — t o or chestr a t e a ne tw ork  o f  
specializ ed agen ts seamlessly  thr ough t ool calls.  I nst ead o f  l...

--- ĐOẠN 2 (Trang 25) ---
Think  o f  guar dr ails as a la y er ed de f ense mechanism.  While a single one is unlik ely  t o pr o vide 
sufficien t pr o t ec tion,  using multiple ,  specializ ed guar dr ails t oge ther  cr e...

--- ĐOẠN 3 (Trang 23) ---
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
),
tools=[track_order_status, initiate_refund_process]
)

triage_agent = Agent(
    name=Triage Agent",
    instructions=
,
    handoffs=[technical_s...
